# Analysis

Analysis selects simple, reproducible evidence about customer themes, sentiment, and support outcomes. It consumes the non-sensitive enriched-ticket handoff and produces tables and visualisations for the final dashboard.

---

## Libraries

Imports the path, numerical, tabular-data, and visualisation libraries used throughout the notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

---

## Data

Loads the non-sensitive enriched-ticket handoff produced by Enrichment.

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "analysis":
    project_root = project_root.parent

analysis_data_dir = project_root / "analysis" / "data"
final_tickets_path = analysis_data_dir / "enriched_tickets.parquet"
if not final_tickets_path.is_file():
    raise FileNotFoundError(
        "Run Enrichment through Export before loading the Analysis input."
    )

analysis_tickets = pd.read_parquet(final_tickets_path)
analysis_input_profile = pd.DataFrame(
    {
        "metric": ["artifact", "tickets", "fields", "themes"],
        "value": [
            final_tickets_path.relative_to(project_root),
            len(analysis_tickets),
            len(analysis_tickets.columns),
            analysis_tickets["theme"].nunique(),
        ],
    }
)

analysis_input_profile

Displays the loaded enriched-ticket dataset before selection or transformation.

In [ ]:
analysis_tickets

---

## Variable Selection

Selects source variables that can explain how enriched themes or sentiment relate to customer outcomes. This notebook does not analyse relationships that exclude both theme and sentiment, because those relationships are outside the purpose of the enrichment pipeline.

In [ ]:
variable_selection = pd.DataFrame(
    [
        ("ticket_date", "Keep", "Measures whether theme share changes over the observed period."),
        ("has_churn", "Keep", "Direct customer outcome for theme and sentiment comparisons."),
        ("resolution_hours", "Keep", "Operational outcome with near-complete coverage."),
        ("customer_message_count", "Keep", "Measures repeated customer contact within a ticket."),
        ("reopen_count", "Keep", "Supporting measure of unresolved or repeated support work."),
        ("csat", "Keep with restriction", "Direct outcome, but only rated tickets support the comparison."),
        ("response_csat", "Keep with restriction", "Identifies whether a CSAT rating exists, not whether the rating is positive."),
        ("main_contact_reason and contact_reason", "Exclude", "Noisy source labels that duplicate the purpose of enriched theme."),
        ("channel, delivery_zone, region, status", "Exclude", "Trivial or fragmented distributions do not support a useful initial comparison."),
        ("business_sla_outcome and event_category", "Exclude", "They do not add a clearer initial customer-outcome view than the selected fields."),
    ],
    columns=["variable", "decision", "reason"],
)

variable_selection

---

## Sentiment Measures

Excludes tickets without an actionable customer issue, then converts each remaining ticket's sentiment list into comparable measures. A score of negative one represents only negative messages, zero represents neutral messages, and one represents only positive messages.

In [ ]:
sentiment_values = {"negative": -1, "neutral": 0, "positive": 1}
excluded_graph_themes = {"No actionable content"}
analysis_frame = analysis_tickets.loc[
    ~analysis_tickets["theme"].isin(excluded_graph_themes)
].copy()
analysis_frame["ticket_date"] = pd.to_datetime(analysis_frame["ticket_date"])
analysis_frame["sentiment_score"] = analysis_frame["sentiments"].map(
    lambda values: pd.Series(values).map(sentiment_values).mean()
)
analysis_frame["negative_share"] = analysis_frame["sentiments"].map(
    lambda values: pd.Series(values).eq("negative").mean()
)
analysis_frame["has_rated_csat"] = analysis_frame["csat"].notna()
analysis_frame["low_csat"] = analysis_frame["csat"].eq(1)
analysis_measure_profile = pd.DataFrame(
    {
        "measure": [
            "Average sentiment score",
            "Average negative-message share",
            "Tickets with a CSAT rating",
        ],
        "value": [
            analysis_frame["sentiment_score"].mean(),
            analysis_frame["negative_share"].mean(),
            analysis_frame["has_rated_csat"].sum(),
        ],
    }
)

analysis_measure_profile

---

## Customer Themes

Summarises the themes customers contact support about and their associated message sentiment. This establishes the baseline volume before comparing the themes with source-system outcomes.

In [ ]:
theme_landscape_summary = (
    analysis_frame.groupby("theme")
    .agg(
        tickets=("ticket_id", "size"),
        average_sentiment=("sentiment_score", "mean"),
        negative_message_share=("negative_share", "mean"),
    )
    .assign(ticket_share=lambda frame: frame["tickets"] / len(analysis_frame))
    .sort_values("tickets", ascending=False)
    .reset_index()
)

theme_landscape_summary

Charts the 12 most frequent themes. Bar length represents ticket count and colour represents average sentiment, so frequent and strongly negative themes are visible together.

In [ ]:
landscape_chart_data = theme_landscape_summary.head(12).sort_values("tickets")
theme_landscape_figure = px.bar(
    landscape_chart_data,
    x="tickets",
    y="theme",
    orientation="h",
    color="average_sentiment",
    color_continuous_scale="RdYlGn",
    range_color=[-1, 1],
    text="tickets",
    title="Customer Themes",
    labels={"average_sentiment": "Average Sentiment Score", "theme": "Theme", "tickets": "Ticket Amount"},
)
theme_landscape_figure.update_traces(textposition="outside")
theme_landscape_figure.update_layout(yaxis={"categoryorder": "total ascending"})
theme_landscape_figure.update_traces(
    hovertemplate="<b>Theme</b>: %{y}<br><b>Ticket Amount</b>: %{x:.0f}<br><b>Average Sentiment Score</b>: %{marker.color:.2f}<extra></extra>"
)

theme_landscape_figure

### Findings

**Meal replacement and order cancellation are the most common reasons customers contact support, with 159 and 125 tickets respectively.** Together, they represent almost three in ten tickets in this sample.

Meal replacement is much more **negative** than order cancellation. Meal damage, incorrect delivery procedure, missing promotion, and meal quality also combine meaningful volume with strongly **negative** customer language.

A **Theme** is the message-based description of what a customer is contacting support about. Bar length and the number beside it show **Ticket Amount**. Colour shows **Average Sentiment Score**, from redder for more **negative** language to greener for more **positive** language.

---

## Theme Trends

Measures daily theme share rather than daily ticket count, so changes in the observed volume do not determine the trend. Themes with fewer than 20 tickets are excluded from this exploratory view because their daily shares are too unstable.

In [ ]:
trend_minimum_tickets = 20
theme_ticket_counts = analysis_frame["theme"].value_counts()
trend_themes = theme_ticket_counts[
    theme_ticket_counts.ge(trend_minimum_tickets)
].index
daily_theme_share = (
    analysis_frame.loc[analysis_frame["theme"].isin(trend_themes)]
    .groupby(["ticket_date", "theme"])
    .size()
    .rename("tickets")
    .reset_index()
)
daily_ticket_counts = analysis_frame.groupby("ticket_date").size().rename("daily_tickets")
daily_theme_share = daily_theme_share.join(daily_ticket_counts, on="ticket_date")
daily_theme_share["share"] = daily_theme_share["tickets"] / daily_theme_share["daily_tickets"]
trend_dates = pd.Series(sorted(analysis_frame["ticket_date"].unique()))
theme_trend_summary = (
    daily_theme_share.groupby("theme")
    .apply(
        lambda group: pd.Series(
            {
                "tickets": theme_ticket_counts[group.name],
                "first_day_share": group.sort_values("ticket_date")["share"].iloc[0],
                "last_day_share": group.sort_values("ticket_date")["share"].iloc[-1],
                "daily_share_change": np.polyfit(
                    (group["ticket_date"] - trend_dates.min()).dt.days,
                    group["share"],
                    1,
                )[0],
            }
        )
    )
    .reset_index()
    .sort_values("daily_share_change", ascending=False)
)

theme_trend_summary

Charts the six eligible themes with the largest absolute daily-share changes over the observed period. The lines are descriptive because the dataset covers only nine dates.

In [ ]:
trend_chart_themes = theme_trend_summary.nlargest(3, "daily_share_change")["theme"].tolist()
falling_themes = theme_trend_summary.nsmallest(3, "daily_share_change")["theme"].tolist()
trend_chart_themes = list(dict.fromkeys(trend_chart_themes + falling_themes))
theme_trend_figure = px.line(
    daily_theme_share.loc[daily_theme_share["theme"].isin(trend_chart_themes)],
    x="ticket_date",
    y="share",
    color="theme",
    markers=True,
    title="Theme Trends",
    labels={"ticket_date": "Ticket Date", "share": "Theme Share", "theme": "Theme"},
)
theme_trend_figure.update_yaxes(tickformat=".0%")
theme_trend_figure.update_traces(
    hovertemplate="<b>Theme</b>: %{fullData.name}<br><b>Ticket Date</b>: %{x|%b %d, %Y}<br><b>Theme Share</b>: %{y:.2%}<extra></extra>"
)

theme_trend_figure

### Findings

**Subscription cancellation, incorrect delivery procedure, and price transparency are becoming more prominent within the observed support contacts.** They are the strongest upward patterns in the period, making them sensible themes to watch in the next reporting window.

Order cancellation, delivery rescheduling, and meal replacement are becoming less prominent over the same period. This does not mean those themes are unimportant, particularly because meal replacement remains the largest overall theme.

The horizontal axis is **Ticket Date** and the vertical axis is **Theme Share**, the percentage of tickets on each date assigned to a theme. Each line is a **Theme**. The view covers nine dates, so it is a directional signal rather than a forecast.

---

## Theme Churn

Compares churn prevalence and average enriched sentiment across themes with at least 20 observed churn outcomes. This keeps each result tied to both an extracted customer theme and a source-system outcome.

In [ ]:
churn_minimum_tickets = 20
theme_churn_summary = (
    analysis_frame.dropna(subset=["has_churn"])
    .groupby("theme")
    .agg(
        tickets=("ticket_id", "size"),
        churn_rate=("has_churn", "mean"),
        average_sentiment=("sentiment_score", "mean"),
    )
    .query("tickets >= @churn_minimum_tickets")
    .sort_values("churn_rate", ascending=False)
    .reset_index()
)

theme_churn_summary

Charts churn rate against average sentiment for the eligible themes. Larger circles represent more tickets, so the strongest rates can be considered with their supporting volume.

In [ ]:
churn_label_themes = {
    "Subscription cancellation",
    "Order cancellation",
    "Meal replacement",
}
theme_churn_chart_data = (
    theme_churn_summary.assign(
        label_theme=lambda frame: frame["theme"].where(
            frame["theme"].isin(churn_label_themes), ""
        )
    )
)
theme_churn_figure = px.scatter(
    theme_churn_chart_data,
    x="average_sentiment",
    y="churn_rate",
    size="tickets",
    text="label_theme",
    custom_data=["theme"],
    size_max=50,
    title="Theme Churn",
    labels={"average_sentiment": "Average Sentiment Score", "churn_rate": "Churn Rate", "tickets": "Tickets"},
)
theme_churn_figure.add_vline(x=analysis_frame["sentiment_score"].mean(), line_dash="dot", line_color="grey")
theme_churn_figure.update_traces(
    textposition="middle center",
    texttemplate="<b>%{text}</b>",
    cliponaxis=False,
)
theme_churn_figure.update_layout(margin={"t": 90, "r": 150, "b": 70, "l": 70})
theme_churn_figure.update_yaxes(tickformat=".0%")
theme_churn_figure.update_traces(
    hovertemplate="<b>Theme</b>: %{customdata[0]}<br><b>Average Sentiment Score</b>: %{x:.2f}<br><b>Churn Rate</b>: %{y:.2%}<br><b>Tickets</b>: %{marker.size:.0f}<extra></extra>"
)

theme_churn_figure

### Findings

**Subscription cancellation has the highest observed churn rate among the larger themes, at 32.0% across 50 known outcomes.** Order cancellation follows at 17.5% across 114 outcomes and meal replacement at 13.8% across 145.

Subscription cancellation is not the most **negative** theme. This suggests that customers leaving may be driven by more than the tone of a single support conversation, and that the cancellation journey is worth examining in its own right.

Each bubble is a **Theme**. Its horizontal position is **Average Sentiment Score**, from minus one for fully **negative** language to one for fully **positive** language. Its vertical position is **Churn Rate**, the percentage of tickets linked to a customer leaving. Bubble size is the number of **Tickets** with a known churn outcome, and the dotted line marks the average sentiment across all tickets.

---

## Resolution Time

Compares median resolution time with average sentiment across themes represented by at least 20 tickets. Medians reduce the influence of unusually long cases while retaining a simple operational measure.

In [ ]:
resolution_minimum_tickets = 20
theme_resolution_summary = (
    analysis_frame.dropna(subset=["resolution_hours"])
    .groupby("theme")
    .agg(
        tickets=("ticket_id", "size"),
        median_resolution_hours=("resolution_hours", "median"),
        average_sentiment=("sentiment_score", "mean"),
    )
    .query("tickets >= @resolution_minimum_tickets")
    .sort_values("median_resolution_hours", ascending=False)
    .reset_index()
)

theme_resolution_summary

Charts median resolution hours for the eligible themes. Colour represents the associated average sentiment, where darker red indicates more negative sentiment.

In [ ]:
resolution_chart_data = theme_resolution_summary.sort_values("median_resolution_hours")
resolution_label_themes = {
    "Missed delivery",
    "Meal quality",
    "Price transparency",
}
resolution_chart_data = resolution_chart_data.assign(
    label_theme=lambda frame: frame["theme"].where(
        frame["theme"].isin(resolution_label_themes), ""
    ),
    label_position="middle left",
)
theme_resolution_figure = px.scatter(
    resolution_chart_data,
    x="median_resolution_hours",
    y="theme",
    color="average_sentiment",
    color_continuous_scale="RdYlGn",
    range_color=[-1, 1],
    size="tickets",
    text="label_theme",
    size_max=35,
    title="Resolution Time",
    labels={"median_resolution_hours": "Median Resolution Hours", "theme": "Theme", "average_sentiment": "Average Sentiment Score"},
)
theme_resolution_figure.update_yaxes(categoryorder="array", categoryarray=resolution_chart_data["theme"].tolist())
theme_resolution_figure.update_traces(
    textposition="middle right",
    texttemplate="<b>%{text}</b>",
    cliponaxis=False,
)
theme_resolution_figure.update_traces(textposition=resolution_chart_data["label_position"])
theme_resolution_figure.update_layout(margin={"t": 70, "r": 180, "b": 70, "l": 190})
theme_resolution_figure.update_traces(
    hovertemplate="<b>Theme</b>: %{y}<br><b>Median Resolution Hours</b>: %{x:.2f}<br><b>Average Sentiment Score</b>: %{marker.color:.2f}<br><b>Tickets</b>: %{marker.size:.0f}<extra></extra>"
)

theme_resolution_figure

### Findings

**Missed delivery, meal quality, and price transparency take the longest to resolve, at 12.4, 12.2, and 11.8 hours respectively.** These are the themes where customers wait the longest for their support case to close.

Meal quality and missed delivery are both slow to resolve and strongly **negative** in customer messages. That combination makes them stronger candidates for an operational experience investigation than a slow theme with mostly neutral language.

Each point is a **Theme**. The horizontal position is **Median Resolution Hours**, the middle resolution time for that theme, so unusually long cases do not dominate the result. Point size is **Tickets** and colour is **Average Sentiment Score**, from redder for more **negative** language to greener for more **positive** language.

---

## Repeat Contact

Compares the average number of customer messages and reopened tickets with the negative-message share for themes represented by at least 20 tickets. Repeated messages remain in the source data because repetition can itself be meaningful for customer experience.

In [ ]:
repeat_minimum_tickets = 20
theme_repeat_summary = (
    analysis_frame.groupby("theme")
    .agg(
        tickets=("ticket_id", "size"),
        average_customer_messages=("customer_message_count", "mean"),
        average_reopens=("reopen_count", "mean"),
        negative_message_share=("negative_share", "mean"),
    )
    .query("tickets >= @repeat_minimum_tickets")
    .sort_values("average_customer_messages", ascending=False)
    .reset_index()
)

theme_repeat_summary

Charts repeated customer messages against negative-message share. Circle size represents the supporting ticket count and the labels keep the view usable as a prioritisation aid.

In [ ]:
repeat_label_themes = {
    "Meal damage",
    "Duplicate order",
    "Price transparency",
    "Meal replacement",
}
theme_repeat_chart_data = theme_repeat_summary.assign(
    label_theme=lambda frame: frame["theme"].where(
        frame["theme"].isin(repeat_label_themes), ""
    )
)
theme_repeat_figure = px.scatter(
    theme_repeat_chart_data,
    x="average_customer_messages",
    y="negative_message_share",
    size="tickets",
    text="label_theme",
    custom_data=["theme"],
    size_max=50,
    title="Repeat Contact",
    labels={"average_customer_messages": "Average Customer Messages", "negative_message_share": "Negative Message Share", "tickets": "Tickets"},
)
theme_repeat_figure.update_traces(
    textposition="middle center",
    texttemplate="<b>%{text}</b>",
    cliponaxis=False,
)
theme_repeat_figure.update_layout(margin={"t": 90, "r": 180, "b": 70, "l": 70})
theme_repeat_figure.update_yaxes(tickformat=".0%")
theme_repeat_figure.update_traces(
    hovertemplate="<b>Theme</b>: %{customdata[0]}<br><b>Average Customer Messages</b>: %{x:.2f}<br><b>Negative Message Share</b>: %{y:.2%}<br><b>Tickets</b>: %{marker.size:.0f}<extra></extra>"
)

theme_repeat_figure

### Findings

**Meal damage, duplicate order, price transparency, and meal replacement combine more back and forth with predominantly negative customer language.** These are the themes in the upper-right area of the chart.

Across the themes shown, more customer messages tend to coincide with a higher share of **negative** language. That makes the themes toward the upper right useful candidates for a closer look at the customer journey.

Each bubble is a **Theme**. The horizontal axis is **Average Customer Messages**, the average number of customer messages in a ticket. The vertical axis is **Negative Message Share**, the percentage of those messages labelled **negative**. Bubble size is **Tickets**.

---

## Low CSAT

Compares the rated CSAT subset with enriched theme and sentiment. `response_csat` is used only to document rating availability because its `unresponded` value is not a customer satisfaction outcome.

In [ ]:
csat_minimum_ratings = 5
theme_csat_summary = (
    analysis_frame.loc[analysis_frame["has_rated_csat"]]
    .groupby("theme")
    .agg(
        rated_tickets=("ticket_id", "size"),
        low_csat_rate=("low_csat", "mean"),
        average_sentiment=("sentiment_score", "mean"),
    )
    .query("rated_tickets >= @csat_minimum_ratings")
    .sort_values("low_csat_rate", ascending=False)
    .reset_index()
)

theme_csat_summary

Charts low-CSAT rate for themes with at least five ratings. The small rating count is displayed beside each bar to prevent overinterpretation.

In [ ]:
csat_chart_data = theme_csat_summary.sort_values("low_csat_rate")
theme_csat_figure = px.bar(
    csat_chart_data,
    x="low_csat_rate",
    y="theme",
    orientation="h",
    custom_data=["rated_tickets"],
    title="Low CSAT",
    labels={"low_csat_rate": "Low CSAT Rate", "theme": "Theme", "rated_tickets": "Rated Tickets"},
    color_discrete_sequence=["#c44e52"],
)
theme_csat_figure.update_xaxes(tickformat=".0%")
theme_csat_figure.update_traces(
    hovertemplate="<b>Theme</b>: %{y}<br><b>Low CSAT Rate</b>: %{x:.2%}<br><b>Rated Tickets</b>: %{customdata[0]:.0f}<extra></extra>"
)

theme_csat_figure

### Findings

**Delivery status has the highest low CSAT rate in the rated subset, at 50.0% across eight ratings.** This is a strong early warning, although the number of ratings is small.

**Meal quality**, **order cancellation**, **meal replacement**, and **payment processing** also show a meaningful share of low ratings. These themes are reasonable to monitor as more survey responses arrive.

Only **150 of 972 tickets** in the analytical sample have a **CSAT rating**, or **15.4%**. This limited coverage means a small number of additional ratings can materially change a theme's result, so the chart is an **early signal** rather than a reliable satisfaction ranking.

Each bar is a **Theme**. Bar length is **Low CSAT Rate**, the percentage of available CSAT ratings equal to one. The label is **Rated Tickets**, the number of responses behind that percentage.